# 09. GRU + LSTM ensemble

**What this notebook does:** adds an LSTM to every fold and averages it with the GRU of the same fold. Notebook 05 showed on fold 0 that an LSTM member helps (0.898 against 0.893 for three GRU seeds), but that ensemble was never scored on all folds or on the leaderboard.

**Members.** The three GRU fold models from notebook 06, and three LSTM fold models trained here with exactly the same recipe (`base_config(fold=k, rnn="lstm")`: same data, seed, vectors, optimizer and decoding). The fold-0 LSTM repeats the `other_rnn` run of notebook 05, which checks that the recipe reproduces on another machine.

**Why the local score is valid now.** Notebook 07 could not score its combination, because each fold was in the training data of two of the three models. Here, the GRU and the LSTM of fold k both trained without fold k, so their average is scored on fold k without a leak. The three fold scores give the CV score of the ensemble.

**Combination.** As decided in notebook 05: log-probabilities and CRF transition scores are averaged, and Viterbi decodes the average once (`combine_models`). Decoding: lenient merge, empty strings dropped, CRF bias 0.

**Inputs.** The `final_cv` folder from notebook 06 (unzip `final_cv.zip` into `data/ner_work/final_cv/`) and the fastText cache `vectors_aligned_200000.npz` in `model/cache/`.

In [1]:
import json
import os
import shutil
import sys
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "eda").exists():
    ROOT = ROOT.parent
sys.path[:0] = [str(ROOT), str(ROOT / "model")]
os.environ.setdefault("NER_WANDB", "off")
os.environ.setdefault("NER_DEVICE", "cpu")

import pandas as pd
from IPython.display import display

from nerkit.artifacts import (combine_models, decode_split, load_logits, majority_vote, row_key,
                              write_submission)
from nerkit.config import ARTIFACTS_DIR, WORK_DIR
from nerkit.data import build_cache, load_labeling_sample, load_test
from nerkit.decisions import DECODING, base_config
from nerkit.experiment import evaluation_sets, records_named, run_experiment, select_training
from nerkit.metrics import evaluate

NOTEBOOK = "09_gru_lstm_ensemble"
FOLDS = (0, 1, 2)
DECODE = {name: DECODING[name] for name in ("merge", "drop_empty", "trim_number", "bias")}
FINAL_CV = ROOT / "data" / "ner_work" / "final_cv"
GRU_KEYS = {fold: base_config(fold=fold).key() for fold in FOLDS}
LSTM_KEYS = {fold: base_config(fold=fold, rnn="lstm").key() for fold in FOLDS}
build_cache()
print("GRU runs:", GRU_KEYS)
print("LSTM runs:", LSTM_KEYS)

GRU runs: {0: '367d29e13b0b', 1: '60d6b400897e', 2: 'c048151b9c64'}
LSTM runs: {0: '83520747a47e', 1: '8854601ee46a', 2: '02224fb3ecf3'}


## The GRU fold models from notebook 06

The saved folders are copied to `artifacts/<run key>/`, where `load_logits` finds them. The run keys in `summary.json` must match the keys of the current decisions, otherwise the files belong to a different recipe.

In [2]:
summary = json.loads((FINAL_CV / "summary.json").read_text(encoding="utf-8"))
for fold, key in GRU_KEYS.items():
    saved_key = summary["folds"][f"fold{fold}"]["run_key"]
    if saved_key != key:
        raise ValueError(f"fold {fold}: final_cv holds run {saved_key}, the current GRU recipe is {key}")
    if not (ARTIFACTS_DIR / key).exists():
        shutil.copytree(FINAL_CV / f"fold{fold}", ARTIFACTS_DIR / key)
print("GRU fold models in place:", sorted(path.name for path in ARTIFACTS_DIR.iterdir() if path.name in GRU_KEYS.values()))

GRU fold models in place: ['367d29e13b0b', '60d6b400897e', 'c048151b9c64']


## Train the LSTM fold models

A finished fold is skipped on rerun. The three folds can also be trained in parallel processes with the same call; the notebook then only reads the finished records.

In [3]:
lstm_records = {fold: run_experiment(f"lstm_fold{fold}", base_config(fold=fold, rnn="lstm"), NOTEBOOK)
                for fold in FOLDS}
reference = records_named("05_decoding_logits_ensembling", ["other_rnn"])[0]
print(f"fold 0 LSTM: F1 mix {lstm_records[0]['scores']['heldout']['mix']['f1']:.3f} here, "
      f"{reference['scores']['heldout']['mix']['f1']:.3f} in notebook 05 (same recipe, other machine)")
display(pd.DataFrame([dict(fold=fold, epochs=len(r["history"]), minutes=r["total_seconds"] / 60,
                           parameters=r["parameters"]) for fold, r in lstm_records.items()]).set_index("fold").round(1))

lstm_fold0: already finished, F1 mix 0.899
lstm_fold1: already finished, F1 mix 0.902
lstm_fold2: already finished, F1 mix 0.906
fold 0 LSTM: F1 mix 0.899 here, 0.899 in notebook 05 (same recipe, other machine)


,epochs,minutes,parameters
fold,,,
0,7,37.7,312450
1,9,46.3,312450
2,10,51.6,312450


## Cross-validation: each fold scored by its own GRU, LSTM and their average

`heldout` is the held-out fold (single sentences); `posts` are synthetic multi-line posts built from the same fold (notebook 04). The GRU rows must reproduce the notebook 06 scores (0.885, 0.896, 0.894), which checks that the copied files load correctly.

In [4]:
def scores(rows, loaded, crf):
    result = evaluate(rows, decode_split(loaded, rows, crf=crf, **DECODE))
    return dict(f1=result["mix"]["f1"], precision=result["mix"]["precision"], recall=result["mix"]["recall"],
                uk_f1=result["uk"]["f1"], ru_f1=result["ru"]["f1"])


def load_pair(fold, split, rows):
    gru, gru_meta = load_logits(GRU_KEYS[fold], split)
    lstm, lstm_meta = load_logits(LSTM_KEYS[fold], split)
    if set(gru) != set(lstm):
        raise ValueError(f"fold {fold}, {split}: GRU and LSTM logits cover different rows; was the cache rebuilt differently?")
    average, average_crf = combine_models([gru, lstm], [gru_meta, lstm_meta], {row_key(r): r["text"] for r in rows})
    return {"GRU": (gru, gru_meta["crf"]), "LSTM": (lstm, lstm_meta["crf"]), "GRU + LSTM average": (average, average_crf)}


cv_rows = []
for fold in FOLDS:
    _, _, _, heldout = select_training(base_config(fold=fold))
    posts = evaluation_sets(base_config(fold=fold), heldout)["posts"]
    heldout_models, posts_models = load_pair(fold, "heldout", heldout), load_pair(fold, "posts", posts)
    for name in heldout_models:
        cv_rows.append(dict(fold=fold, model=name, **scores(heldout, *heldout_models[name]),
                            posts_f1=scores(posts, *posts_models[name])["f1"]))
cv = pd.DataFrame(cv_rows)
display(cv.pivot(index="fold", columns="model", values="f1").round(3))
cv_mean = cv.groupby("model")[["f1", "precision", "recall", "uk_f1", "ru_f1", "posts_f1"]].agg(["mean", "std"])
display(cv_mean.round(3))

model,GRU,GRU + LSTM average,LSTM
fold,,,
0,0.885,0.898,0.899
1,0.896,0.906,0.902
2,0.894,0.907,0.906


f1        precision        recall         uk_f1         \
                     mean    std      mean    std   mean    std   mean    std   
model                                                                           
GRU                 0.891  0.006     0.911  0.004  0.873  0.013  0.883  0.008   
GRU + LSTM average  0.904  0.005     0.920  0.004  0.888  0.006  0.896  0.006   
LSTM                0.902  0.004     0.911  0.010  0.894  0.003  0.895  0.005   

                    ru_f1        posts_f1         
                     mean    std     mean    std  
model                                             
GRU                 0.947  0.005    0.873  0.009  
GRU + LSTM average  0.955  0.004    0.892  0.004  
LSTM                0.952  0.006    0.890  0.004

## Test predictions of all six models

On the competition posts every model is valid, so all six are combined. Three files are written:
- `ensemble_gru_lstm_average.csv`: the average of all six models (the main candidate),
- `ensemble_lstm_average.csv`: the three LSTMs alone, so the LSTM also has a leaderboard score next to its CV score,
- `ensemble_gru_lstm_vote3.csv`: a string is kept when at least 3 of the 6 models predict it (notebook 07's vote did slightly better than the average on the leaderboard).

The 26 labeled competition posts are scored as a rough check only; they are too few to choose between methods.

In [5]:
test_rows, labeling_rows = load_test(), load_labeling_sample()


def combined_predictions(keys, rows, split):
    loaded = [load_logits(key, split) for key in keys]
    average, crf = combine_models([logits for logits, _ in loaded], [meta for _, meta in loaded],
                                  {row_key(r): r["text"] for r in rows})
    return decode_split(average, rows, crf=crf, **DECODE)


def single_predictions(key, rows, split):
    logits, meta = load_logits(key, split)
    return decode_split(logits, rows, crf=meta["crf"], **DECODE)


ALL_KEYS = [*GRU_KEYS.values(), *LSTM_KEYS.values()]
METHODS = ("GRU average (notebook 07)", "LSTM average", "GRU + LSTM average", "GRU + LSTM vote, 3 of 6")


def predict(method, rows, split):
    if method == "GRU average (notebook 07)":
        return combined_predictions(GRU_KEYS.values(), rows, split)
    if method == "LSTM average":
        return combined_predictions(LSTM_KEYS.values(), rows, split)
    if method == "GRU + LSTM average":
        return combined_predictions(ALL_KEYS, rows, split)
    if method == "GRU + LSTM vote, 3 of 6":
        return majority_vote([single_predictions(key, rows, split) for key in ALL_KEYS], 3)
    raise ValueError(f"Unknown method {method}")


test_predictions, overview = {}, []
for name in METHODS:
    test_predictions[name] = predict(name, test_rows, "test")
    labeling = evaluate(labeling_rows, predict(name, labeling_rows, "labeling"))["pooled"]
    overview.append(dict(method=name, posts_with_locations=sum(bool(p) for p in test_predictions[name]),
                         locations=sum(len(p) for p in test_predictions[name]),
                         labeling_precision=labeling["precision"], labeling_recall=labeling["recall"],
                         labeling_f1=labeling["f1"]))
display(pd.DataFrame(overview).set_index("method").round(3))

,posts_with_locations,locations,labeling_precision,labeling_recall,labeling_f1
method,,,,,
GRU average (notebook 07),322,734,0.625,0.781,0.694
LSTM average,334,778,0.575,0.719,0.639
GRU + LSTM average,327,752,0.571,0.750,0.649
"GRU + LSTM vote, 3 of 6",332,768,0.571,0.750,0.649


In [6]:
FILES = {"LSTM average": "ensemble_lstm_average.csv", "GRU + LSTM average": "ensemble_gru_lstm_average.csv",
         "GRU + LSTM vote, 3 of 6": "ensemble_gru_lstm_vote3.csv"}
for name, file in FILES.items():
    path = write_submission(WORK_DIR / "submissions" / file, test_rows, test_predictions[name])
    print(f"{path.name}: {sum(bool(p) for p in test_predictions[name])} of {len(test_rows)} posts have locations")

ensemble_lstm_average.csv: 334 of 477 posts have locations
ensemble_gru_lstm_average.csv: 327 of 477 posts have locations
ensemble_gru_lstm_vote3.csv: 332 of 477 posts have locations


## Leaderboard results

In [7]:
LEADERBOARD = {  # public F1, copied from Kaggle; earlier rows from notebooks 06 and 07
    "GRU, best single fold (notebook 06)": 0.58594,
    "GRU average (notebook 07)": 0.58787,
    "GRU vote, 2 of 3 (notebook 07)": 0.58914,
    "LSTM average": 0.56304,
    "GRU + LSTM average": 0.57698,
    "GRU + LSTM vote, 3 of 6": 0.57311,
}
display(pd.Series(LEADERBOARD, name="public_f1").to_frame())

,public_f1
"GRU, best single fold (notebook 06)",0.58594
GRU average (notebook 07),0.58787
"GRU vote, 2 of 3 (notebook 07)",0.58914
LSTM average,0.56304
GRU + LSTM average,0.57698
"GRU + LSTM vote, 3 of 6",0.57311


**Conclusion.** The leaderboard does not confirm the cross-validation gain. On held-out folds the LSTM beats the GRU (0.902 against 0.891) and the GRU + LSTM average is the best (0.904). On the competition posts the order is reversed: the three LSTMs alone score 0.563, the GRU + LSTM average 0.577, and the GRU-only average of notebook 07 0.588. The 26 labeled competition posts show the same order (GRU 0.694, LSTM 0.639, GRU + LSTM 0.649). So the LSTM fits the source sentences better but transfers worse to Telegram posts, and the final submission stays the GRU vote of notebook 07 (0.58914).